# ViText2SQL — Zero-shot & Few-shot với API (DeepSeek, Gemini)

Notebook này chạy **Text-to-SQL** trên ViText2SQL bằng mô hình API (không cần GPU local):
- **DeepSeek** (`deepseek-chat`, `deepseek-coder`)
- **Google Gemini** (`gemini-2.0-flash`, `gemini-1.5-pro`, `gemini-1.5-flash`)

Hỗ trợ hai chế độ:
1. **Zero-shot**: chỉ schema + câu hỏi
2. **Few-shot**: thêm ví dụ (BM25 hoặc random từ **train set**)

**Đánh giá trên tập TEST** (theo paper): câu hỏi từ `test.json`, SQL gốc từ `test_gold.sql`.
Metrics: **Exact Match** và **Component F1** (chuẩn Spider/ViText2SQL).

## 1. Cài đặt dependencies

In [ ]:
# Chạy cell này nếu chưa cài thư viện
# !pip install openai google-generativeai rank-bm25 sqlparse tqdm pandas

## 2. Cấu hình

In [ ]:
import os
import sys
from pathlib import Path

# Thêm thư mục gốc project vào PYTHONPATH
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========== CẤU HÌNH — Sửa các giá trị bên dưới ==========

# Mô hình API: deepseek-chat | deepseek-coder | gemini-2.0-flash | gemini-1.5-pro | gemini-1.5-flash
MODEL_KEY = "deepseek-coder"

# Chế độ: "zero_shot" hoặc "few_shot"
MODE = "few_shot"

# Dữ liệu: word-level hoặc syllable-level
DATA_DIR = PROJECT_ROOT / "data" / "word-level"

# Tập đánh giá: "test" (mặc định, theo paper) hoặc "dev"
SPLIT = "test"

# Few-shot: ví dụ lấy từ train.json (không dùng test/dev)
NUM_SHOTS = 3
EXAMPLE_STRATEGY = "bm25"  # "bm25" hoặc "random"

# Giới hạn mẫu (None = chạy hết test set; đặt số nhỏ để thử nhanh)
MAX_SAMPLES = 10

# Tham số API
TEMPERATURE = 0.1
MAX_TOKENS = 1024

# API Keys — điền trực tiếp HOẶC đặt biến môi trường
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "") or os.environ.get("GOOGLE_API_KEY", "")

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "api_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Model: {MODEL_KEY} | Mode: {MODE} | Split: {SPLIT} | Data: {DATA_DIR.name}")

## 3. Tải dữ liệu ViText2SQL

In [ ]:
from src.utils import (
    load_eval_split,
    build_tables_index,
    build_text2sql_prompt,
    schema_linking,
    normalize_predicted_sql,
    save_json,
    select_few_shot_examples_bm25,
    select_few_shot_examples_random,
)
from src.api_clients import create_api_client, API_MODEL_REGISTRY

eval_data, train_data, tables_data = load_eval_split(DATA_DIR, split=SPLIT)
tables_index = build_tables_index(tables_data)

if MAX_SAMPLES:
    eval_data = eval_data[:MAX_SAMPLES]

print(f"Split: {SPLIT} | Train (few-shot pool): {len(train_data)} mẫu | Đánh giá: {len(eval_data)} mẫu")
print(f"Mô hình API khả dụng: {list(API_MODEL_REGISTRY.keys())}")

## 4. Khởi tạo client API

In [ ]:
# Chọn API key theo provider
provider = API_MODEL_REGISTRY[MODEL_KEY]["provider"]
api_key = DEEPSEEK_API_KEY if provider == "deepseek" else GEMINI_API_KEY

if not api_key:
    raise ValueError(
        f"Thiếu API key cho {provider}.\n"
        "- DeepSeek: https://platform.deepseek.com/ → DEEPSEEK_API_KEY\n"
        "- Gemini: https://aistudio.google.com/apikey → GEMINI_API_KEY"
    )

client = create_api_client(
    model_key=MODEL_KEY,
    api_key=api_key,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

print(f"Đã kết nối: {API_MODEL_REGISTRY[MODEL_KEY]['description']}")

## 5. Thử nhanh một câu hỏi (demo)

In [ ]:
sample = eval_data[0]
db_id = sample["db_id"]
question = sample["question"]
gold_sql = sample["query"]
table_entry = tables_index[db_id]

few_shot_examples = None
if MODE == "few_shot":
    if EXAMPLE_STRATEGY == "bm25":
        few_shot_examples = select_few_shot_examples_bm25(
            target_question=question,
            candidate_pool=train_data,
            num_examples=NUM_SHOTS,
            target_db_id=db_id,
        )
    else:
        few_shot_examples = select_few_shot_examples_random(
            candidate_pool=train_data,
            num_examples=NUM_SHOTS,
            target_db_id=db_id,
            exclude_question=question,
        )

reference_sql = few_shot_examples[0]["sql"] if few_shot_examples else None
schema_text = schema_linking(table_entry, question, reference_sql=reference_sql)
prompt = build_text2sql_prompt(question, schema_text, few_shot_examples)

print("=" * 60)
print("PROMPT GỬI ĐẾN API:")
print("=" * 60)
print(prompt[:2000] + ("..." if len(prompt) > 2000 else ""))
print("\n" + "=" * 60)

raw_response = client.generate_with_retry(prompt)
predicted_sql = normalize_predicted_sql(raw_response)

print(f"Câu hỏi:  {question}")
print(f"Gold SQL: {gold_sql}")
print(f"Pred SQL: {predicted_sql}")

## 6. Chạy inference trên tập TEST (hoặc dev nếu SPLIT="dev")

In [ ]:
import time
from tqdm.notebook import tqdm

predictions = []
errors = []

for item in tqdm(eval_data, desc=f"{MODE} | {MODEL_KEY}"):
    db_id = item["db_id"]
    question = item["question"]
    gold_sql = item["query"]
    table_entry = tables_index[db_id]

    few_shot_examples = None
    if MODE == "few_shot":
        if EXAMPLE_STRATEGY == "bm25":
            few_shot_examples = select_few_shot_examples_bm25(
                target_question=question,
                candidate_pool=train_data,
                num_examples=NUM_SHOTS,
                target_db_id=db_id,
            )
        else:
            few_shot_examples = select_few_shot_examples_random(
                candidate_pool=train_data,
                num_examples=NUM_SHOTS,
                target_db_id=db_id,
                exclude_question=question,
            )

    reference_sql = few_shot_examples[0]["sql"] if few_shot_examples else None
    schema_text = schema_linking(table_entry, question, reference_sql=reference_sql)
    prompt = build_text2sql_prompt(question, schema_text, few_shot_examples)

    try:
        raw_response = client.generate_with_retry(prompt)
        predicted_sql = normalize_predicted_sql(raw_response)
    except Exception as error:
        predicted_sql = ""
        errors.append({"question": question, "error": str(error)})

    predictions.append({
        "db_id": db_id,
        "question": question,
        "gold_sql": gold_sql,
        "predict_sql": predicted_sql,
        "gold_query_toks": item.get("query_toks"),
        "mode": MODE,
        "model": MODEL_KEY,
        "example_strategy": EXAMPLE_STRATEGY if MODE == "few_shot" else None,
    })

    # Tránh rate limit
    time.sleep(0.5)

output_file = OUTPUT_DIR / f"predictions_{MODEL_KEY}_{MODE}_{SPLIT}_{DATA_DIR.name}.json"
save_json(predictions, output_file)

print(f"\nHoàn tất: {len(predictions)} mẫu, {len(errors)} lỗi")
print(f"Đã lưu: {output_file}")

## 7. Đánh giá EM & Component F1

In [ ]:
from src.spider_eval.evaluation_core import evaluate_batch
import pandas as pd

tables_path = DATA_DIR / "tables.json"
metrics = evaluate_batch(predictions, tables_path=str(tables_path))

metrics_file = OUTPUT_DIR / f"metrics_{MODEL_KEY}_{MODE}_{SPLIT}_{DATA_DIR.name}.json"
save_json(metrics, metrics_file)

df_metrics = pd.DataFrame([{
    "Mô hình": MODEL_KEY,
    "Chế độ": MODE,
    "Split": SPLIT,
    "Dữ liệu": DATA_DIR.name,
    "Số mẫu": metrics["total_samples"],
    "Exact Match": f"{metrics['exact_match']*100:.2f}%",
    "SELECT F1": f"{metrics['select_f1']*100:.2f}%",
    "WHERE F1": f"{metrics['where_f1']*100:.2f}%",
    "GROUP F1": f"{metrics['group_f1']*100:.2f}%",
    "ORDER F1": f"{metrics['order_f1']*100:.2f}%",
    "Avg F1": f"{metrics['average_component_f1']*100:.2f}%",
}])
df_metrics

## 8. Xem các mẫu sai (debug)

In [ ]:
from src.spider_eval.evaluation_core import evaluate_em_f1
from src.spider_eval.process_sql import Schema, build_schema_from_tables_entry

wrong_samples = []

for item in predictions:
    table_entry = tables_index[item["db_id"]]
    schema = Schema(build_schema_from_tables_entry(table_entry))
    result = evaluate_em_f1(
        predict_sql=item["predict_sql"],
        gold_sql=item["gold_sql"],
        db_schema=schema,
        gold_tokens=item.get("gold_query_toks"),
    )
    if result["exact_match"] == 0:
        wrong_samples.append({
            "question": item["question"],
            "gold_sql": item["gold_sql"],
            "predict_sql": item["predict_sql"],
            "avg_f1": result["average_component_f1"],
        })

print(f"Số mẫu sai EM: {len(wrong_samples)} / {len(predictions)}")
pd.DataFrame(wrong_samples[:5])

## 9. So sánh Zero-shot vs Few-shot (tùy chọn)

Chạy cell dưới để benchmark nhanh cả hai chế độ trên cùng số mẫu.

In [ ]:
def run_benchmark(mode: str, max_samples: int = 5) -> dict:
    """Chạy benchmark nhanh trên tập SPLIT hiện tại."""
    subset = eval_data[:max_samples]
    preds = []

    for item in subset:
        db_id = item["db_id"]
        question = item["question"]
        table_entry = tables_index[db_id]

        few_shot_examples = None
        if mode == "few_shot":
            few_shot_examples = select_few_shot_examples_bm25(
                question, train_data, NUM_SHOTS, target_db_id=db_id
            )

        ref = few_shot_examples[0]["sql"] if few_shot_examples else None
        schema_text = schema_linking(table_entry, question, reference_sql=ref)
        prompt = build_text2sql_prompt(question, schema_text, few_shot_examples)

        raw = client.generate_with_retry(prompt)
        preds.append({
            "db_id": db_id,
            "gold_sql": item["query"],
            "predict_sql": normalize_predicted_sql(raw),
            "gold_query_toks": item.get("query_toks"),
        })
        time.sleep(0.5)

    return evaluate_batch(preds, tables_path=str(DATA_DIR / "tables.json"))


COMPARE_SAMPLES = 5  # Số mẫu so sánh

results = []
for mode in ["zero_shot", "few_shot"]:
    print(f"Đang chạy {mode} trên {SPLIT}...")
    m = run_benchmark(mode, max_samples=COMPARE_SAMPLES)
    results.append({
        "Split": SPLIT,
        "Mode": mode,
        "EM": f"{m['exact_match']*100:.1f}%",
        "Avg F1": f"{m['average_component_f1']*100:.1f}%",
    })

pd.DataFrame(results)